# LoRA Experiment Board

This notebook is part of the Atlas Math Lab research workbench.

Goals:
- Compare profile configs across adapters, runtimes, and datasets.
- Summarize trainable ratios and optimization settings.
- Use the board to decide which experiment family to run next.


In [ ]:
from __future__ import annotations

import sys
from pathlib import Path

ROOT = Path.cwd()
if (ROOT / "notebooks").exists():
    REPO_ROOT = ROOT
else:
    REPO_ROOT = ROOT.parent

if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

REPO_ROOT

### Inspecting LoRA Profiles

This section scans the `profiles` directory for experiment configurations. We extract key hyperparameter choices like `run_name`, `seed`, and potentially LoRA rank or learning rates to build a comprehensive view of all planned or completed experiments.


In [ ]:
import pandas as pd
import yaml

profile_dir = REPO_ROOT / "training" / "configs" / "profiles"
rows = []
if profile_dir.exists():
    for path in sorted(profile_dir.glob("*.yaml")):
        try:
            payload = yaml.safe_load(path.read_text())
            rows.append(
                {
                    "profile": path.stem,
                    "run_name": payload.get("run_name", "N/A"),
                    "seed": payload.get("seed", "N/A"),
                    "lora_r": payload.get("lora_config", {}).get("r", "N/A"),
                    "learning_rate": payload.get("training_arguments", {}).get("learning_rate", "N/A"),
                }
            )
        except Exception as e:
            print(f"Error loading {path.name}: {e}")

df_profiles = pd.DataFrame(rows)
if not df_profiles.empty:
    display(df_profiles)
else:
    print("No profiles found.")

### Comparing Configurations

By analyzing the extracted dataframe, we can visually compare different experiment settings, helping us understand which hyperparameter combinations have been explored.


In [ ]:
if not df_profiles.empty and "lora_r" in df_profiles.columns:
    print("Distribution of LoRA ranks:")
    print(df_profiles["lora_r"].value_counts())